# WMDP Target-Model Full GPU Test

This notebook runs the real target model on the full local WMDP bio/chem/cyber datasets with no corruption. It is the next gate after `kaggle_wmdp_target_model_mini_gpu.ipynb` passed with `sample_size=2`.

Defaults: `PORT_TARGET_CONFIG_NAME=phi-1_5`, `PORT_ATTN_IMPLEMENTATION=eager`, `PORT_TORCH_DTYPE=float16`, `PORT_WMDP_BATCH_SIZE=1`.


In [1]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys
import time

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()

def has_project_layout(path):
    path = Path(path)
    return (path / 'llm-unlearn-eco' / 'eco').exists() and (path / 'dataset').exists()

def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()

PROJECT_ROOT = clone_or_use_project()
ECO_ROOT = PROJECT_ROOT / 'llm-unlearn-eco'
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(ECO_ROOT) not in sys.path:
    sys.path.insert(0, str(ECO_ROOT))

commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


Cloning https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git into /kaggle/working/PoRT_LLM_Unlearning-Experiment


Cloning into '/kaggle/working/PoRT_LLM_Unlearning-Experiment'...


Project root: /kaggle/working/PoRT_LLM_Unlearning-Experiment
Commit: 1bf20f22e5c8f34d546f04b152148d2ec3f1ad4c


In [2]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'tabulate': 'tabulate',
    'tqdm': 'tqdm',
    'transformers': 'transformers>=4.38.0',
    'accelerate': 'accelerate>=0.26.0',
    'yaml': 'pyyaml',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


Required packages are already available.


In [3]:
import gc
import platform
import pandas as pd
import torch
import yaml

from eco.dataset import WMDPBio, WMDPChem, WMDPCyber
from eco.evaluator import ChoiceByTopLogit
from eco.inference import EvaluationEngine
from eco.model import HFModel
from eco.paths import MODEL_CONFIG_DIR

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is required for this full target-model test. Enable a Kaggle GPU runtime and rerun.')

for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f'GPU {idx}: {props.name}, VRAM={props.total_memory / 1024**3:.2f} GiB')


Python: 3.12.13
Torch: 2.10.0+cu128
CUDA available: True
GPU 0: Tesla T4, VRAM=14.56 GiB
GPU 1: Tesla T4, VRAM=14.56 GiB


## Runtime Model Config


In [4]:
def env_bool(name, default=None):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

TARGET_CONFIG_NAME = os.environ.get('PORT_TARGET_CONFIG_NAME', 'phi-1_5')
TARGET_HF_NAME = os.environ.get('PORT_TARGET_HF_NAME')
TARGET_MODEL_PATH = os.environ.get('PORT_TARGET_MODEL_PATH') or None
ATTN_IMPLEMENTATION = os.environ.get('PORT_ATTN_IMPLEMENTATION', 'eager')
TORCH_DTYPE = os.environ.get('PORT_TORCH_DTYPE', 'float16')
BATCH_SIZE = int(os.environ.get('PORT_WMDP_BATCH_SIZE', '1'))
SAMPLE_SIZE = None

base_config_path = MODEL_CONFIG_DIR / f'{TARGET_CONFIG_NAME}.yaml'
if not base_config_path.exists():
    raise FileNotFoundError(f'Model config not found: {base_config_path}')

with open(base_config_path, 'r', encoding='utf-8') as f:
    runtime_config = yaml.safe_load(f)

if TARGET_HF_NAME:
    runtime_config['hf_name'] = TARGET_HF_NAME
runtime_config['attn_implementation'] = ATTN_IMPLEMENTATION
runtime_config['torch_dtype'] = TORCH_DTYPE

load_in_8bit = env_bool('PORT_LOAD_IN_8BIT', runtime_config.get('load_in_8bit', False))
load_in_4bit = env_bool('PORT_LOAD_IN_4BIT', runtime_config.get('load_in_4bit', False))
runtime_config['load_in_8bit'] = bool(load_in_8bit)
runtime_config['load_in_4bit'] = bool(load_in_4bit)

RUN_NAME = f'wmdp_target_full_no_corrupt_{TARGET_CONFIG_NAME}'
RUN_DIR = (Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT / 'results') / RUN_NAME
RUNTIME_CONFIG_DIR = RUN_DIR / 'model_config'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_MODEL_NAME = f'{TARGET_CONFIG_NAME}_runtime_gpu_full'
runtime_config_path = RUNTIME_CONFIG_DIR / f'{RUNTIME_MODEL_NAME}.yaml'
with open(runtime_config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(runtime_config, f, sort_keys=False)

run_config = {
    'repo_url': REPO_URL,
    'commit_sha': commit_sha,
    'target_config_name': TARGET_CONFIG_NAME,
    'target_hf_name': runtime_config.get('hf_name'),
    'target_model_path': TARGET_MODEL_PATH,
    'runtime_model_name': RUNTIME_MODEL_NAME,
    'runtime_config_path': str(runtime_config_path),
    'attn_implementation': ATTN_IMPLEMENTATION,
    'torch_dtype': TORCH_DTYPE,
    'load_in_8bit': runtime_config['load_in_8bit'],
    'load_in_4bit': runtime_config['load_in_4bit'],
    'batch_size': BATCH_SIZE,
    'sample_size': SAMPLE_SIZE,
    'corrupt_method': None,
    'datasets': ['wmdp-bio', 'wmdp-chem', 'wmdp-cyber'],
    'run_dir': str(RUN_DIR),
}
(RUN_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
print(json.dumps(run_config, indent=2))


{
  "repo_url": "https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git",
  "commit_sha": "1bf20f22e5c8f34d546f04b152148d2ec3f1ad4c",
  "target_config_name": "phi-1_5",
  "target_hf_name": "microsoft/phi-1_5",
  "target_model_path": null,
  "runtime_model_name": "phi-1_5_runtime_gpu_full",
  "runtime_config_path": "/kaggle/working/wmdp_target_full_no_corrupt_phi-1_5/model_config/phi-1_5_runtime_gpu_full.yaml",
  "attn_implementation": "eager",
  "torch_dtype": "float16",
  "load_in_8bit": false,
  "load_in_4bit": false,
  "batch_size": 1,
  "sample_size": null,
  "corrupt_method": null,
  "datasets": [
    "wmdp-bio",
    "wmdp-chem",
    "wmdp-cyber"
  ],
  "run_dir": "/kaggle/working/wmdp_target_full_no_corrupt_phi-1_5"
}


## Load Target Model


In [5]:
start_model = time.perf_counter()
model = HFModel(
    model_name=RUNTIME_MODEL_NAME,
    model_path=TARGET_MODEL_PATH,
    config_path=str(RUNTIME_CONFIG_DIR),
)
if model.tokenizer.pad_token_id is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
model.tokenizer.padding_side = 'right'
model_load_seconds = time.perf_counter() - start_model
print(f'Model loaded in {model_load_seconds:.2f}s')
if hasattr(model.model, 'hf_device_map'):
    print('HF device map:', json.dumps(model.model.hf_device_map, default=str, indent=2))
if torch.cuda.is_available():
    print('CUDA memory allocated GiB:', torch.cuda.memory_allocated() / 1024**3)
    print('CUDA memory reserved GiB:', torch.cuda.memory_reserved() / 1024**3)


config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.84G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

Number of parameters: 1418270720


tokenizer_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Model loaded in 16.14s
HF device map: {
  "model.embed_tokens": 0,
  "model.layers.0": 0,
  "model.layers.1": 0,
  "model.layers.2": 0,
  "model.layers.3": 0,
  "model.layers.4": 0,
  "model.layers.5": 0,
  "model.layers.6": 0,
  "model.layers.7": 0,
  "model.layers.8": 0,
  "model.layers.9": 0,
  "model.layers.10": 0,
  "model.layers.11": 1,
  "model.layers.12": 1,
  "model.layers.13": 1,
  "model.layers.14": 1,
  "model.layers.15": 1,
  "model.layers.16": 1,
  "model.layers.17": 1,
  "model.layers.18": 1,
  "model.layers.19": 1,
  "model.layers.20": 1,
  "model.layers.21": 1,
  "model.layers.22": 1,
  "model.layers.23": 1,
  "model.rotary_emb": 1,
  "model.embed_dropout": 1,
  "model.final_layernorm": 1,
  "lm_head": 1
}
CUDA memory allocated GiB: 1.2270240783691406
CUDA memory reserved GiB: 1.228515625


## Run Full WMDP Dataset, No-Corrupt


In [6]:
choice_labels = ['A', 'B', 'C', 'D']
all_summaries = []
all_predictions = []
subject_timings = {}
completed_datasets = []

def write_partial_artifacts():
    partial_predictions = pd.DataFrame(all_predictions)
    if len(partial_predictions):
        partial_predictions.to_csv(RUN_DIR / 'predictions_partial.csv', index=False)
    partial_payload = {
        'run_config': run_config,
        'model_load_seconds': model_load_seconds,
        'subject_timings_seconds': subject_timings,
        'completed_datasets': completed_datasets,
        'engine_summaries': all_summaries,
        'total_rows_so_far': int(len(partial_predictions)),
    }
    (RUN_DIR / 'summary_partial.json').write_text(json.dumps(partial_payload, indent=2, default=str), encoding='utf-8')

for data_module in [WMDPBio(), WMDPChem(), WMDPCyber()]:
    print(f'\nRunning {data_module.name} full target-model test...')
    subject_start = time.perf_counter()
    engine = EvaluationEngine(
        model=model,
        tokenizer=model.tokenizer,
        data_module=data_module,
        subset_names=['test'],
        evaluator=ChoiceByTopLogit(),
        batch_size=BATCH_SIZE,
    )
    engine.inference()
    summary_stats, outputs = engine.summary()
    elapsed = time.perf_counter() - subject_start
    subject_timings[data_module.name] = elapsed
    all_summaries.extend(summary_stats)

    dataset_predictions = []
    result_name, batches = list(outputs[0].items())[0]
    row_idx = 0
    for batch in batches:
        for c, p in zip(batch['correct'], batch['predicted']):
            row = {
                'dataset': data_module.name,
                'row_index': row_idx,
                'correct_index': int(c),
                'predicted_index': int(p),
                'correct_label': choice_labels[int(c)] if 0 <= int(c) < len(choice_labels) else str(c),
                'predicted_label': choice_labels[int(p)] if 0 <= int(p) < len(choice_labels) else str(p),
                'is_correct': int(c) == int(p),
            }
            dataset_predictions.append(row)
            all_predictions.append(row)
            row_idx += 1

    completed_datasets.append(data_module.name)
    pd.DataFrame(dataset_predictions).to_csv(RUN_DIR / f'predictions_{data_module.name}.csv', index=False)
    write_partial_artifacts()
    print(f'{data_module.name} completed in {elapsed:.2f}s with {row_idx} rows.')

predictions_df = pd.DataFrame(all_predictions)
summary_by_dataset = predictions_df.groupby('dataset')['is_correct'].agg(['count', 'mean']).reset_index()
summary_by_dataset = summary_by_dataset.rename(columns={'mean': 'accuracy'})
print(summary_by_dataset)



Running wmdp-bio full target-model test...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1273 [00:00<?, ? examples/s]

Map:   0%|          | 0/1273 [00:00<?, ? examples/s]

Evaluating choice_by_top_logit of wmdp-bio on test: 100%|██████████| 1273/1273 [00:59<00:00, 21.28it/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1987 [00:00<?, ? examples/s]

Map:   0%|          | 0/1987 [00:00<?, ? examples/s]

Evaluating choice_by_top_logit of wmdp-cyber on test: 100%|██████████| 1987/1987 [04:00<00:00,  8.27it/s]

{'wmdp-cyber_test_choice_by_top_logit': 0.3241066935078007}
wmdp-cyber completed in 241.24s with 1987 rows.
      dataset  count  accuracy
0    wmdp-bio   1273  0.523959
1   wmdp-chem    408  0.335784
2  wmdp-cyber   1987  0.324107


In [7]:
summary_payload = {
    'run_config': run_config,
    'model_load_seconds': model_load_seconds,
    'subject_timings_seconds': subject_timings,
    'engine_summaries': all_summaries,
    'summary_by_dataset': summary_by_dataset.to_dict(orient='records'),
    'total_rows': int(len(predictions_df)),
    'overall_accuracy': float(predictions_df['is_correct'].mean()) if len(predictions_df) else None,
}

summary_path = RUN_DIR / 'summary.json'
predictions_path = RUN_DIR / 'predictions.csv'
summary_by_dataset_path = RUN_DIR / 'summary_by_dataset.csv'
summary_path.write_text(json.dumps(summary_payload, indent=2, default=str), encoding='utf-8')
predictions_df.to_csv(predictions_path, index=False)
summary_by_dataset.to_csv(summary_by_dataset_path, index=False)

print(f'Wrote {summary_path}')
print(f'Wrote {predictions_path}')
print(f'Wrote {summary_by_dataset_path}')
print(json.dumps(summary_payload, indent=2, default=str)[:4000])


Wrote /kaggle/working/wmdp_target_full_no_corrupt_phi-1_5/summary.json
Wrote /kaggle/working/wmdp_target_full_no_corrupt_phi-1_5/predictions.csv
Wrote /kaggle/working/wmdp_target_full_no_corrupt_phi-1_5/summary_by_dataset.csv
{
  "run_config": {
    "repo_url": "https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git",
    "commit_sha": "1bf20f22e5c8f34d546f04b152148d2ec3f1ad4c",
    "target_config_name": "phi-1_5",
    "target_hf_name": "microsoft/phi-1_5",
    "target_model_path": null,
    "runtime_model_name": "phi-1_5_runtime_gpu_full",
    "runtime_config_path": "/kaggle/working/wmdp_target_full_no_corrupt_phi-1_5/model_config/phi-1_5_runtime_gpu_full.yaml",
    "attn_implementation": "eager",
    "torch_dtype": "float16",
    "load_in_8bit": false,
    "load_in_4bit": false,
    "batch_size": 1,
    "sample_size": null,
    "corrupt_method": null,
    "datasets": [
      "wmdp-bio",
      "wmdp-chem",
      "wmdp-cyber"
    ],
    "run_dir": "/kaggle/working/wmdp_targ

In [8]:
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('WMDP TARGET-MODEL FULL GPU TEST COMPLETED')


WMDP TARGET-MODEL FULL GPU TEST COMPLETED
